# Tilt census

A tilt-distance-specific census for anticyclonic (AE) and cyclonic (CE) eddies. The notebook retains the original mirrored-histogram idea but uses compact publication dashboards, smooth shading and embedded weighted box summaries.

Daily distributions give every eddy equal total weight, preventing long tracks from dominating. Per-eddy metrics contain one observation per eddy.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.ticker import FuncFormatter
import seacofs_tilt_tools as tilt

AE_COLOUR = '#C44E52'
CE_COLOUR = '#2878A5'
TYPE_COLOURS = {'AE': AE_COLOUR, 'CE': CE_COLOUR}
PAGE_SIZE = (11.7, 8.3)
SAVE_FIGURES = False
FIGURE_DIR = Path('tilt_census_figures')
SMOOTH_SIGMA_BINS = 1.15
UPSTREAM_REGIONS = ['S1', 'U1', 'U2']
DOWNSTREAM_REGIONS = ['S2', 'D1', 'D2']
SUBREGION_ORDER = ['S1', 'U1', 'U2', 'S2', 'D1', 'D2']

mpl.rcParams.update({'figure.dpi':120,'savefig.dpi':600,'font.size':9.5,
 'axes.titlesize':10.5,'axes.labelsize':9.5,'legend.fontsize':8.5,
 'axes.spines.top':False,'axes.spines.right':False,'axes.linewidth':.8,
 'xtick.direction':'out','ytick.direction':'out','pdf.fonttype':42,'ps.fonttype':42})

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df_eddies, df_tilt = tilt.load_tilt_tables(paths, add_regions=True, grid=grid)
df_eddies = df_eddies.sort_values(['Cyc','Eddy','Day']).copy()
region_group_map = {'S1':'Shelf','S2':'Shelf','U1':'Upstream','U2':'Upstream','D1':'Downstream','D2':'Downstream'}
sector_map = {**{r:'Upstream + S1' for r in UPSTREAM_REGIONS}, **{r:'Downstream + S2' for r in DOWNSTREAM_REGIONS}}
df_eddies['RegionGroup'] = df_eddies.Region.map(region_group_map)
df_eddies['Sector'] = df_eddies.Region.map(sector_map)
KEYS = ['Cyc','Eddy']
df_eddies['day_index'] = df_eddies.groupby(KEYS).cumcount()
last_index = df_eddies.groupby(KEYS).day_index.transform('max')
df_eddies['norm_time'] = np.where(last_index>0,df_eddies.day_index/last_index,np.nan)
df_eddies['lifetime_days'] = df_eddies.groupby(KEYS).Day.transform(lambda x:x.max()-x.min()+1)
tilt_data = df_eddies.dropna(subset=['TiltDis']).query("Cyc in ['AE','CE']").copy()
GLOBAL_TILT_LIMIT = float(tilt_data.TiltDis.quantile(.995))
GLOBAL_TILT_BINS = np.linspace(0,GLOBAL_TILT_LIMIT,43)
df_eddies.head()

## Census summary
Counts remain available for documentation, while the figures below focus on tilt-distance results.

In [ ]:
census = (df_eddies.groupby(['Cyc','Sector'],dropna=False)
          .agg(eddy_days=('Eddy','size'),unique_eddies=('Eddy','nunique'),
               tilt_days=('TiltDis','count'),median_tilt=('TiltDis','median')).reset_index())
display(census.round(2))
for cyc in ['AE','CE']:
    g=df_eddies[df_eddies.Cyc==cyc]; eligible=len(g)-6*g.Eddy.nunique()
    print(f'{cyc}: {g.Eddy.nunique():,} eddies; {g.TiltDis.notna().sum():,} tilt estimates; '
          f'{100*g.TiltDis.notna().sum()/eligible:.1f}% eligible-day success')

## Plotting and per-eddy summaries

In [ ]:
def equal_eddy_weights(data):
    return 1.0/data.groupby(KEYS).Eddy.transform('size').astype(float)

def weighted_quantile(values,quantiles,weights=None):
    values=np.asarray(values,float); weights=np.ones(values.size) if weights is None else np.asarray(weights,float)
    ok=np.isfinite(values)&np.isfinite(weights)&(weights>0); values,weights=values[ok],weights[ok]
    if not values.size: return np.full(np.atleast_1d(quantiles).shape,np.nan)
    order=np.argsort(values); values,weights=values[order],weights[order]
    cdf=(np.cumsum(weights)-.5*weights)/weights.sum()
    return np.interp(np.atleast_1d(quantiles),cdf,values)

def gaussian_smooth(y,sigma=SMOOTH_SIGMA_BINS):
    radius=max(1,int(np.ceil(4*sigma))); x=np.arange(-radius,radius+1)
    kernel=np.exp(-.5*(x/sigma)**2); kernel/=kernel.sum()
    return np.convolve(y,kernel,mode='same')

def make_eddy_summary(data):
    rows=[]
    for (cyc,eddy),g in data.groupby(KEYS):
        x=g.TiltDis.dropna().to_numpy()
        if not len(x): continue
        rows.append({'Cyc':cyc,'Eddy':eddy,'mean_tilt':x.mean(),'median_tilt':np.median(x),
          'min_tilt':x.min(),'max_tilt':x.max(),'tilt_range':np.ptp(x),
          'tilt_sd':np.std(x,ddof=1) if len(x)>1 else np.nan,'p90_tilt':np.quantile(x,.9),
          'n_tilt':len(x),'lifetime_days':g.lifetime_days.max()})
    return pd.DataFrame(rows)

def draw_weighted_box(ax,values,weights,y,colour):
    q05,q25,q50,q75,q95=weighted_quantile(values,[.05,.25,.5,.75,.95],weights)
    ax.plot([q05,q95],[y,y],color=colour,lw=1.2,solid_capstyle='round')
    ax.plot([q05,q05],[y-.10,y+.10],color=colour,lw=1); ax.plot([q95,q95],[y-.10,y+.10],color=colour,lw=1)
    ax.add_patch(Rectangle((q25,y-.18),q75-q25,.36,facecolor=colour,edgecolor=colour,alpha=.24,lw=1.1))
    ax.plot([q50,q50],[y-.18,y+.18],color=colour,lw=2)

def mirrored_density_box(ax,data,column,title,xlabel,*,daily=False,bins=None,xlim=None,show_legend=False):
    clean=data.dropna(subset=[column]).copy()
    if bins is None:
        values=clean[column].to_numpy(); hi=np.nanquantile(values,.995)
        bins=np.linspace(max(0,np.nanmin(values)),hi,38)
    centres=.5*(bins[:-1]+bins[1:]); stored={}
    for cyc,sign in [('AE',1),('CE',-1)]:
        g=clean[clean.Cyc==cyc]; values=g[column].to_numpy()
        weights=equal_eddy_weights(g).to_numpy() if daily else np.ones(len(g))
        hist,_=np.histogram(values,bins=bins,weights=weights); hist=hist/hist.sum()*100 if hist.sum() else hist.astype(float)
        smooth=gaussian_smooth(hist); colour=TYPE_COLOURS[cyc]
        ax.fill_between(centres,0,sign*smooth,color=colour,alpha=.30,lw=0)
        ax.plot(centres,sign*smooth,color=colour,lw=1.8,label=cyc); stored[cyc]=(values,weights)
    ax.axhline(0,color='.25',lw=.7); ax.yaxis.set_major_formatter(FuncFormatter(lambda y,_:f'{abs(y):g}'))
    ax.set(title=title,xlabel=xlabel,ylabel='Probability (%)'); ax.grid(axis='x',alpha=.14,lw=.6)
    if xlim is not None: ax.set_xlim(xlim)
    box=ax.inset_axes([.05,.79,.92,.18],sharex=ax)
    draw_weighted_box(box,*stored['AE'],.70,AE_COLOUR); draw_weighted_box(box,*stored['CE'],.30,CE_COLOUR)
    box.set_ylim(0,1); box.set_yticks([]); box.tick_params(axis='x',labelbottom=False,bottom=False)
    for spine in box.spines.values(): spine.set_visible(False)
    box.patch.set_alpha(0)
    if show_legend: ax.legend(frameon=False,ncol=2,loc='lower right')

def panel_label(ax,label):
    ax.text(-.10,1.06,label,transform=ax.transAxes,fontweight='bold',fontsize=11)

def finish_figure(fig,name,title):
    fig.suptitle(title,fontsize=14,fontweight='bold',y=.997)
    if SAVE_FIGURES:
        FIGURE_DIR.mkdir(exist_ok=True); fig.savefig(FIGURE_DIR/f'{name}.pdf',bbox_inches='tight')
        fig.savefig(FIGURE_DIR/f'{name}.png',bbox_inches='tight',dpi=600)
    plt.show()

overall_eddy=make_eddy_summary(tilt_data)

## Figure 1 — Compact overall tilt census
Six magnitude summaries replace the original tall figure. Each inset shows the weighted 5th–95th percentile whisker, IQR and median.

In [ ]:
metrics=[
 ('TiltDis','All tilt observations','Tilt distance (km)',True,tilt_data),
 ('mean_tilt','Mean tilt per eddy','Mean tilt (km)',False,overall_eddy),
 ('median_tilt','Median tilt per eddy','Median tilt (km)',False,overall_eddy),
 ('min_tilt','Minimum tilt per eddy','Minimum tilt (km)',False,overall_eddy),
 ('max_tilt','Maximum tilt per eddy','Maximum tilt (km)',False,overall_eddy),
 ('tilt_range','Within-eddy tilt range','Maximum − minimum (km)',False,overall_eddy)]
fig,axes=plt.subplots(2,3,figsize=PAGE_SIZE,constrained_layout=True)
for i,(ax,(column,title,xlabel,daily,source)) in enumerate(zip(axes.flat,metrics)):
    mirrored_density_box(ax,source,column,title,xlabel,daily=daily,bins=GLOBAL_TILT_BINS if daily else None,show_legend=i==0)
    panel_label(ax,chr(97+i))
finish_figure(fig,'01_overall_tilt_census','Overall AE and CE tilt-distance census')

## Figure 2 — Upstream and downstream census
The upstream sector is S1 + U1 + U2; the downstream sector is S2 + D1 + D2. Daily tilt uses common bins.

In [ ]:
sector_specs=[('Upstream: S1 + U1 + U2',UPSTREAM_REGIONS),('Downstream: S2 + D1 + D2',DOWNSTREAM_REGIONS)]
fig,axes=plt.subplots(3,2,figsize=PAGE_SIZE,constrained_layout=True)
for j,(sector_title,regions) in enumerate(sector_specs):
    daily=tilt_data[tilt_data.Region.isin(regions)].copy(); eddy=make_eddy_summary(daily)
    mirrored_density_box(axes[0,j],daily,'TiltDis',sector_title,'Tilt distance (km)',daily=True,bins=GLOBAL_TILT_BINS,show_legend=j==0)
    mirrored_density_box(axes[1,j],eddy,'median_tilt','Median tilt per eddy','Median tilt (km)')
    mirrored_density_box(axes[2,j],eddy,'max_tilt','Maximum tilt per eddy','Maximum tilt (km)')
for i,ax in enumerate(axes.flat): panel_label(ax,chr(97+i))
finish_figure(fig,'02_upstream_downstream','Regional contrast in AE and CE tilt distance')

## Figure 3 — Six-region tilt atlas
Compact small multiples retain S1–D2 rather than pooling away local differences.

In [ ]:
fig,axes=plt.subplots(2,3,figsize=PAGE_SIZE,sharex=True,sharey=True,constrained_layout=True)
for i,(ax,region) in enumerate(zip(axes.flat,SUBREGION_ORDER)):
    subset=tilt_data[tilt_data.Region==region]
    mirrored_density_box(ax,subset,'TiltDis',region,'Tilt distance (km)',daily=True,bins=GLOBAL_TILT_BINS,show_legend=i==0)
    panel_label(ax,chr(97+i))
finish_figure(fig,'03_six_region_atlas','Tilt-distance distributions across the six regions')

## Figure 4 — Lifetime and lifecycle alternatives
Lifetime classes are AE/CE-specific eddy-level terciles. Lifecycle stages use normalised time.

In [ ]:
eddy_life=tilt_data.groupby(KEYS,as_index=False).lifetime_days.max(); life_frames=[]
life_labels=['Short-lived','Intermediate','Long-lived']
for cyc,g in eddy_life.groupby('Cyc'):
    q1,q2=g.lifetime_days.quantile([1/3,2/3]); g=g.copy()
    g['LifetimeClass']=pd.cut(g.lifetime_days,[-np.inf,q1,q2,np.inf],labels=life_labels); life_frames.append(g)
life_map=pd.concat(life_frames,ignore_index=True); life_data=tilt_data.merge(life_map,on=KEYS+['lifetime_days'],how='left')
stage_data=tilt_data.copy(); stage_data['Stage']=pd.cut(stage_data.norm_time,[-.001,1/3,2/3,1.001],labels=['Early','Middle','Late'])
fig,axes=plt.subplots(2,3,figsize=PAGE_SIZE,sharex=True,sharey=True,constrained_layout=True)
for j,label in enumerate(life_labels):
    subset=life_data[life_data.LifetimeClass==label]
    mirrored_density_box(axes[0,j],subset,'TiltDis',label,'Tilt distance (km)',daily=True,bins=GLOBAL_TILT_BINS,show_legend=j==0)
for j,label in enumerate(['Early','Middle','Late']):
    subset=stage_data[stage_data.Stage==label]
    mirrored_density_box(axes[1,j],subset,'TiltDis',label+' lifecycle','Tilt distance (km)',daily=True,bins=GLOBAL_TILT_BINS)
for i,ax in enumerate(axes.flat): panel_label(ax,chr(97+i))
finish_figure(fig,'04_lifetime_lifecycle','Tilt distance by eddy lifetime and lifecycle stage')

## Interpretation and export notes

- Solid curves and shading show the smoothed mirrored histogram; the underlying bins remain fixed and reproducible.
- The inset box summaries are weighted consistently with their histograms.
- Set SAVE_FIGURES=True to export editable vector PDFs and 600 dpi PNGs.
- Add eddy-level bootstrap confidence intervals only after selecting the preferred dashboard.